# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL (FAIR^2 dataset)
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)
meta = dataset.metadata

# Print out basic dataset info
print(f"{meta.name}: {meta.description}")
print(f"Published: {meta.datePublished}")
print(f"Version: {getattr(meta, 'version', 'N/A')}")
print(f"Identifier: {meta.identifier}")
print(f"Keywords: {', '.join(meta.keywords)}\n")

## 2. Data Overview
Review available record sets, their fields, and columns. All references are by entity `@id`.

List the dataset's record sets, their `@id`s, and their top-level fields/columns.

In [ ]:
# List all record sets and their fields/columns by @id
record_sets = [r for r in dataset.metadata.recordSets]
print(f"Found {len(record_sets)} record sets:\n")
for rs in record_sets:
    print(f"RecordSet: {rs.name} (@id: {rs.id})")
    print(f"  Description: {getattr(rs, 'description', '')}")
    # List fields and columns
    fields = getattr(rs, 'fields', [])
    columns = getattr(rs, 'columns', [])
    if len(fields):
        print(f"  Fields:")
        for f in fields:
            print(f"    - {f.name} (@id: {f.id}, type: {getattr(f, 'dataType', 'N/A')})")
    if len(columns):
        print(f"  Columns:")
        for c in columns:
            print(f"    - {c.name} (@id: {c.id}, type: {getattr(c, 'dataType', 'N/A')})")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

We use the record set and field `@id`s identified above. All data access is by `@id`.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.recordSets]
print(f"Record set IDs: {record_set_ids}\n")

# Load each record set into a DataFrame and store by @id
dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading records for: {rs_id}")
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"  Loaded {len(df)} records, columns: {df.columns.tolist()}\n")
    except Exception as e:
        print(f"  Could not load records for {rs_id}: {e}\n")

# Select first available record set to preview (you may change this to a specific set below)
if record_set_ids:
    preview_id = record_set_ids[0]
    print(f"Columns in {preview_id}: {dataframes[preview_id].columns.tolist()}")
    dataframes[preview_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps, such as filtering, normalization, and grouping, referencing field/column `@id`s.

In [ ]:
# Select a record set @id to work with (replace with your relevant @id as determined above, if different)
record_set_id = record_set_ids[0]  # Example: use the first one
df = dataframes[record_set_id].copy()
print(f"Working with record set: {record_set_id}\n")

# List available columns (fields/column @id)
print(f"Data columns (by Croissant @id): {df.columns.tolist()}\n")

# Choose a numeric field for analysis (replace as needed)
# Try to automatically find a numeric field by simple heuristic, otherwise set by hand
numeric_field_id = None
for c in df.columns:
    if df[c].dtype in ['float64','int64']:
        numeric_field_id = c
        break
if not numeric_field_id:
    # Try to cast columns
    for c in df.columns:
        try:
            _ = df[c].astype(float)
            numeric_field_id = c
            df[c] = df[c].astype(float)
            break
        except:
            continue
if not numeric_field_id:
    raise ValueError("No numeric column found for demo. Please inspect df.columns and pick a numeric field.")
print(f"Using numeric field '@id': {numeric_field_id}\n")

# Filter example (e.g., values greater than a threshold)
threshold = df[numeric_field_id].mean() if not pd.isna(df[numeric_field_id].mean()) else 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a qualitative field, e.g. the first string column
group_field = None
for col in df.columns:
    if df[col].dtype == 'object' and col != numeric_field_id:
        group_field = col
        break

if group_field and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame()
    print(f"\nGrouped data by {group_field} (mean of {numeric_field_id}):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the selected numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f"Distribution for: {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# If grouping was successful, plot group means
if group_field and group_field in filtered_df.columns:
    grouped_means = filtered_df.groupby(group_field)[numeric_field_id].mean()
    grouped_means = grouped_means.sort_values(ascending=False).head(10)
    plt.figure(figsize=(10,5))
    sns.barplot(x=grouped_means.index, y=grouped_means.values)
    plt.title(f"Mean {numeric_field_id} by {group_field} (top 10)")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we've loaded the FAIR^2 dataset using its Croissant schema, explored available record sets and fields using their `@id`, and performed basic analysis and visualization.

- The dataset describes proteins identified via mass spectrometry from human mast cell extracellular vesicles.
- All data processing referenced fields and record sets via their `@id`, ensuring schema-consistent access.
- You can adapt this workflow to further analyze or integrate related Croissant datasets.

For additional fields or record sets, repeat the analysis above using their respective `@id`s. Refer to the official [mlcroissant documentation](https://mlcommons.org/croissant/) for advanced operations.
